PROBLEMS TO BE SOLVED:
- [ ] The `target_pmfs` of C51 is only normalized if all of `next_pmfs > 0`. If some of them are zero then `target_pmfs.sum() < 1`. This is because `dml_scalar` and `dmu_scalar` do not take the probabilities of `next_pmfs` into account (i.e., they scale everything regardless if its probability happens to be zero).

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
from jax import numpy as jnp # JAX version of NumPy.
from flax import nnx # Neural networks in JAX.
import numpy as np # For compatibility with gymnasium.
import pennylane as qml

import qnrl

jax.config.update("jax_enable_x64", True) # Enable double-precision (default is single-precision).

In [ ]:
### Here we pull out some results seen during training to re-create the system without needing to train again.

num_atoms = 2**5
z_min = -100
z_max = 100
atoms = jnp.linspace(start=z_min, stop=z_max, num=num_atoms)

b = jnp.array([0.31, 1.3, 2.29, 3.28, 4.27, 5.26, 6.25, 7.24, 8.23, 9.22, 10.21, 11.2, 12.19, 13.18, 14.17, 15.16, 16.15, 17.14, 18.13, 19.12, 20.11, 21.1, 22.09, 23.08, 24.07, 25.06, 26.05, 27.04, 28.03, 29.02, 30.01, 31.], dtype=float)
l = jnp.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 30], dtype=int)
u = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 31], dtype=int)

# Atom axis is normalized: dml_scalar[i] + dmu_scalar[i] = 1.
dml_scalar = jnp.array([6.90000000e-01, 7.00000000e-01, 7.10000000e-01, 7.20000000e-01, 7.30000000e-01, 7.40000000e-01, 7.50000000e-01, 7.60000000e-01, 7.70000000e-01, 7.80000000e-01, 7.90000000e-01, 8.00000000e-01, 8.10000000e-01, 8.20000000e-01, 8.30000000e-01, 8.40000000e-01, 8.50000000e-01, 8.60000000e-01, 8.70000000e-01, 8.80000000e-01, 8.90000000e-01, 9.00000000e-01, 9.10000000e-01, 9.20000000e-01, 9.30000000e-01, 9.40000000e-01, 9.50000000e-01, 9.60000000e-01, 9.70000000e-01, 9.80000000e-01, 9.90000000e-01, 7.10542736e-15])
dmu_scalar = jnp.array([0.31, 0.3, 0.29, 0.28, 0.27, 0.26, 0.25, 0.24, 0.23, 0.22, 0.21, 0.2, 0.19, 0.18, 0.17, 0.16, 0.15, 0.14, 0.13, 0.12, 0.11, 0.1, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01, 1.])

target_pmfs = jnp.array([0.02115465, 0.02905183, 0.03054183, 0.03305032, 0.03253098, 0.03309612, 0.03054828, 0.03085908, 0.03039246, 0.02902569, 0.03254757, 0.03105026, 0.03207069, 0.03591443, 0.03094733, 0.03524417, 0.03091764, 0.03371455, 0.03022025, 0.03212222, 0.03066229, 0.02539635, 0.02968344, 0.03383755, 0.02991721, 0.0327772, 0.03349901, 0.03143953, 0.03354148, 0.03228632, 0.03055001, 0.03140925])

assert l.shape == u.shape == dml_scalar.shape == dmu_scalar.shape
assert target_pmfs.sum() - 1 <= 1e-5 # Check if close to 1 using tolerance due to rounding errors in JAX.

Given return-atom PMF state of the form: $pmf_{next} = \sum_{z} \alpha_{z} | z \rangle$

Recipe for $U$:

1. Multiply $pmf_{next}$ twice to compute spread across lower/upper bins:
   - $dml = dml_{scalar} \cdot pmf_{next}$ = $\big(\sum_{z} s^{(l)}_{z} | z \rangle \langle z |\big) \cdot pmf_{next}$ = $\sum_{z} s^{(l)}_{z} \alpha_{z} | z \rangle$
   - $dmu = dmu_{scalar} \cdot pmf_{next}$ = $\big(\sum_{z} s^{(u)}_{z} | z \rangle \langle z |\big) \cdot pmf_{next}$ = $\sum_{z} s^{(u)}_{z} \alpha_{z} | z \rangle$
2. Construct $pmf_{target}$ by adding using indices $l$ and $u$ separately ($l$ maps an atom $z_{in} \to z_{l}$, and $u$ maps $z_{in} \to z_{u}$):
   - $pmf_{target} = \big( \sum_{(z_{in}, z_{l}) \in l} | z_{l} \rangle \langle z_{in} | \big) \cdot dml + \big( \sum_{(z_{in}, z_{u}) \in u} | z_{u} \rangle \langle z_{in} | \big) \cdot dmu$

In [11]:
atoms

Array([-100.        ,  -93.5483871 ,  -87.09677419,  -80.64516129,
        -74.19354839,  -67.74193548,  -61.29032258,  -54.83870968,
        -48.38709677,  -41.93548387,  -35.48387097,  -29.03225806,
        -22.58064516,  -16.12903226,   -9.67741935,   -3.22580645,
          3.22580645,    9.67741935,   16.12903226,   22.58064516,
         29.03225806,   35.48387097,   41.93548387,   48.38709677,
         54.83870968,   61.29032258,   67.74193548,   74.19354839,
         80.64516129,   87.09677419,   93.5483871 ,  100.        ],      dtype=float64)

In [40]:
# Initial U is all zeros, as we will add elements later.
# U = jnp.zeros((l.shape[0], l.shape[0]))
U = jnp.zeros((l.shape[0],))


for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
    # print(f"{z_in, (z_l.item(), z_u.item(), s_l.item(), s_u.item()) = }")
    
    # U = U.at[z_l.astype(jnp.int32), z_in].add(jnp.sqrt(s_l))
    # U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))
    
    U = U.at[z_l.astype(jnp.int32)].add(s_l)
    U = U.at[z_u.astype(jnp.int32)].add(s_u)

nnx.display(U)

In [ ]:

psi = jnp.zeros((num_atoms,1))
# psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi.at[:,-1].set(1)
psi = psi / jnp.count_nonzero(psi.astype(int))

print(psi.flatten())

U @ psi

[0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.        ]


Array([0.99967742], dtype=float64)

In [32]:
# Test U by applying it to an input PMF.

psi = jnp.zeros((num_atoms,1))
psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi / jnp.sqrt(jnp.count_nonzero(psi.astype(int)))

print(psi.flatten())

jnp.linalg.norm((U @ psi))

[0.4472136 0.4472136 0.        0.4472136 0.        0.4472136 0.
 0.4472136 0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.       ]


Array(2.84551771, dtype=float64)

In [22]:
qubits = 1
zero = nnx.one_hot(0, num_classes=2**qubits).reshape((2**qubits,1))
one = nnx.one_hot(1, num_classes=2**qubits).reshape((2**qubits,1))

# jnp.outer(zero, one)
zero @ one.T

Array([[0., 1.],
       [0., 0.]], dtype=float64)

-----

In [4]:
total_scalar = jnp.zeros_like(dml_scalar)
total_scalar = total_scalar.at[l].add(dml_scalar)
total_scalar = total_scalar.at[u].add(dmu_scalar)

# total_scalar


# Compute rotation angles for scalar values.
# `sqrt(dml_scalar)` is `cos` value and `sqrt(dmu_scalar)` is `sin` value.
thetas_scalar = jnp.acos(jnp.sqrt(dml_scalar)) # Rotation angles to produce the `dml_scalar` and `dmu_scalar` multiples using `cos` and `sin` respectively.

jnp.cos(thetas_scalar) # sqrt(dml_scalar)
jnp.sin(thetas_scalar) # sqrt(dmu_scalar)

jnp.exp(1j*thetas_scalar)

Array([8.30662386e-01+0.55677644j, 8.36660027e-01+0.54772256j,
       8.42614977e-01+0.53851648j, 8.48528137e-01+0.52915026j,
       8.54400375e-01+0.51961524j, 8.60232527e-01+0.50990195j,
       8.66025404e-01+0.5j       , 8.71779789e-01+0.48989795j,
       8.77496439e-01+0.47958315j, 8.83176087e-01+0.46904158j,
       8.88819442e-01+0.45825757j, 8.94427191e-01+0.4472136j ,
       9.00000000e-01+0.43588989j, 9.05538514e-01+0.42426407j,
       9.11043358e-01+0.41231056j, 9.16515139e-01+0.4j       ,
       9.21954446e-01+0.38729833j, 9.27361850e-01+0.37416574j,
       9.32737905e-01+0.36055513j, 9.38083152e-01+0.34641016j,
       9.43398113e-01+0.33166248j, 9.48683298e-01+0.31622777j,
       9.53939201e-01+0.3j       , 9.59166305e-01+0.28284271j,
       9.64365076e-01+0.26457513j, 9.69535971e-01+0.24494897j,
       9.74679434e-01+0.2236068j , 9.79795897e-01+0.2j       ,
       9.84885780e-01+0.17320508j, 9.89949494e-01+0.14142136j,
       9.94987437e-01+0.1j       , 8.42936970e-08+1.j  

In [5]:
# Attempt to make operator U using dml/dmu as real/imag of complex.

U = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')

for i, (b_i, l_i, u_i) in enumerate(zip(b,l,u)):
    # print(f"{i, (b_i, l_i, u_i) = }")
    
    # U = U.at[i, i].add(jnp.sqrt(dml_scalar[i]) + 1j*jnp.sqrt(dmu_scalar[i]))
    
    U = U.at[l_i.astype(jnp.int32), l_i.astype(jnp.int32)].add(jnp.sqrt(dml_scalar[i]))
    U = U.at[u_i.astype(jnp.int32), u_i.astype(jnp.int32)].add(1j*jnp.sqrt(dmu_scalar[i]))

nnx.display(U)
# nnx.display(jnp.abs(U))
# nnx.display(U, U.conj().T, U @ U.conj().T)
qnrl.tools.is_unitary(U)

Array(False, dtype=bool)

In [6]:
# Attempt to build an operator U from b, l, u, dml, dmu.

U = jnp.zeros((l.shape[0], l.shape[0]), dtype='float')

for i, (b_i, l_i, u_i) in enumerate(zip(b,l,u)):
    # print(f"{i, (b_i, l_i, u_i) = }")
    
    U = U.at[i, l_i.astype(jnp.int32)].add(jnp.sqrt(dml_scalar[i]))
    U = U.at[i, u_i.astype(jnp.int32)].add(jnp.sqrt(dmu_scalar[i]))

nnx.display(U, U.conj().T, U @ U.conj().T)

In [7]:
# Try making reflection of off-diagonal elements negative.
M = U.copy()
M_inv = jnp.linalg.inv(U.conj().T)
# M = M + U - U.T # Create a negative variant around the reflection.
# M = M + jnp.diag(jnp.diag(U)) # Re-insert the original diagonal around the negative reflection.
nnx.display(M_inv @ M_inv.conj().T)

In [8]:
# M = jnp.zeros_like(U)
# M = M.at[0,0].add(1)
# atoms_probs_toencode = [(0,0.3), (5,0.7)]
atoms_probs_toencode = [(0,0.5), (1,0.5)]
psi = jnp.zeros((num_atoms,1))
for z, p in atoms_probs_toencode:
    # psi = psi + nnx.one_hot(z, num_classes=num_atoms).reshape((num_atoms,1))
    psi = psi + jnp.sqrt(p)*nnx.one_hot(z, num_classes=num_atoms).reshape((num_atoms,1))
# psi = psi / jnp.sqrt(len(atoms_to_encode))
# psi = nnx.one_hot(0, num_classes=num_atoms).reshape((num_atoms,1)) + nnx.one_hot(5, num_classes=num_atoms).reshape((num_atoms,1))

print(psi.flatten())
jnp.linalg.norm((U.T @ psi))

[0.70710678 0.70710678 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]


Array(1.21071573, dtype=float64)

In [9]:
psi = jnp.zeros((num_atoms,1))
psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi / jnp.sqrt(jnp.count_nonzero(psi.astype(int)))

print(psi.flatten())

jnp.linalg.norm((U.T @ psi))

[0.4472136 0.4472136 0.        0.4472136 0.        0.4472136 0.
 0.4472136 0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.       ]


Array(1.08918916, dtype=float64)

In [10]:
nnx.display(U @ U.conj().T)